# 42장 · RAG 화면의 출처 추적과 회귀 검사

검색은 올바른 문서를 찾고도 화면에 지난 질문의 출처가 남을 수 있습니다. 같은 질문의 도구 결과와 최종 표시값을 이어서 확인합니다.

‘답은 14일인데 근거는 휴관 안내’라는 결함을 진단합니다. 검색 직후의 문서 ID, 답변에 붙은 인용, 화면에 표시된 쪽 번호를 한 요청에서 대조합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 검색 결과와 표시 출처 대조

결함을 재현할 때는 입력과 답변을 고정합니다. 질문을 고정한 뒤 검색기가 돌려준 ID, 답이 인용한 ID, 화면이 실제로 띄운 출처를 순서대로 읽습니다. 아래는 불일치를 재현한 수업용 화면 기록입니다.


### 셀 01 · 검색 직후 문서 ID 확인

자료 세 쪽에서 질문 한 건을 다시 실행해 처음 찾은 문서를 읽습니다.

화면에 보여 줄 출처는 이번 질문에서 받은 문서만 사용하세요. 검색 결과가 비었거나 답에 붙은 인용 ID가 검색 문서와 다르면 현재 출처를 비우고 확인을 요청합니다. 지난 질문의 출처를 대신 붙이지 않습니다.


**실행 준비** · 30장의 문서 세 건과 retrieve를 불러옵니다. 아래 준비 셀을 한 번 실행한 뒤 이어지는 예제에서 이번 장의 변경을 확인하세요.

In [ ]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]


In [1]:
question = '대출 기간은 며칠인가요?'
found = retrieve(question, 1)
if found:
    print(found[0]['id'], found[0]['text'])
else:
    print('이번 질문의 검색 결과가 없습니다.')

library-1 도서 대출 기간은 14일입니다.


검색 직후에는 library-1과 14일이 맞아떨어집니다. 검색 입력을 바꾸기 전에 답과 화면 사이를 살펴볼 차례입니다.

답이 이상할 때는 먼저 질문과 검색 ID를 한 쌍으로 적어 두세요. library-1의 본문이 14일을 말하는데 화면에는 다른 쪽이 표시된다면 검색을 다시 조정하기 전에 현재 응답과 출처 패널을 잇는 코드를 볼 수 있습니다.


### 셀 02 · 이전 응답의 출처 잔존 검사

같은 질문의 검색 ID와 화면의 이전 출처를 대조하세요.


In [2]:
screen = {'answer': '도서 대출 기간은 14일입니다. [library-1]', 'source': records[1]}
print(screen['source']['id'], screen['source']['page'])

library-2 2


화면은 library-2의 2쪽을 가리킵니다. 답의 인용과 검색은 대출인데 화면에만 지난 휴관 문서가 남았습니다.


#### 답변은 맞지만 출처 표시가 틀릴 때

<mark>검색이 반환한 문서와 화면이 선택한 문서를 같은 질문으로 대조합니다.</mark> 휴관 답변 옆의 쪽 번호가 대출 문서를 가리킨다면 검색부터 다시 짤 필요는 없습니다. 화면이 이전 결과의 첫 문서를 붙였는지 확인하세요.

선수 개념 · 40장 Streamlit 폼으로 만드는 RAG 질의 화면

### 같은 요청에서 처음 달라진 값

<p>검색 결과와 답의 인용은 모두 library-1인데 화면의 source만 library-2입니다. 화면에 남아 있는 records[1]을 새 검색 결과와 대조하세요. 여기서 질의나 검색 설정까지 바꾸면 출처를 고친 뒤 같은 조건으로 비교하기 어렵습니다.</p>


## 동일 질의로 출처 수정 검증

검색은 같은 대출 문서를 반환합니다. 화면에서 출처를 고르는 코드만 수정하고 같은 질문으로 다시 확인하세요.


### 과제 03 · 현재 검색 결과로 출처 갱신

답에서 인용을 읽고 모두 이번 검색 결과에 있는지 확인하세요. 맞으면 이 예제의 첫 인용을 화면에 연결하세요. 문서가 없거나 인용이 빠지거나 틀리면 출처를 비우고 답 자리에는 안내를 표시하세요.


In [3]:
# 답변에 현재 문서의 ID가 실제로 인용됐는지 확인한 뒤 교체하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>import re

def match_source(answer, documents):
    cited = re.findall(r&quot;\[([^\[\]]+)\]&quot;, answer)
    by_id = {doc[&quot;id&quot;]: doc for doc in documents}
    if cited and all(key in by_id for key in cited):
        return by_id[cited[0]]
    return None

source = match_source(screen[&quot;answer&quot;], found)
if source is None:
    screen = {&quot;answer&quot;: &quot;이번 답의 근거를 확인할 수 없습니다.&quot;, &quot;source&quot;: None}
    print(screen[&quot;answer&quot;])
else:
    screen = {**screen, &quot;source&quot;: source}
    print(source[&quot;id&quot;], source[&quot;page&quot;])</code></pre>
<p>실행 결과</p><pre><code>library-1 1</code></pre>
<p>기본 질문에서는 library-1 1이 보입니다. 셀 01 질문을 ‘주차 요금은?’으로 바꾸고 01~03을 다시 실행하세요. 이번 답의 근거를 확인할 수 없다는 문구가 나오고 screen[&quot;source&quot;]는 None이 됩니다.</p>
</details>


### 과제 04 · 잘못된 인용이 섞인 응답의 회귀 검사

과제 03에서 작성한 match_source를 사용하세요. 대출·휴관 두 문서를 함께 인용한 응답을 대출 문서 한 건만 있는 검색 결과와 대조하세요. 첫 인용이 유효해도 잘못된 출처를 표시하지 않는지 검사하세요.


In [ ]:
answer = '기간은 14일입니다. [library-1] [library-2]'
# 직접 작성한 match_source에 현재 검색 목록과 답변을 전달하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>answer = &#x27;기간은 14일입니다. [library-1] [library-2]&#x27;
current_docs = retrieve(&#x27;대출 기간은?&#x27;, 1)
source = match_source(answer, current_docs)
assert source is None
print(source)</code></pre>
<p>실행 결과</p><pre><code>None</code></pre>
<p>library-1은 맞지만 library-2는 이번 검색 결과에 없습니다. 인용 하나만 확인하고 1쪽을 표시하는 구현은 이 회귀 검사를 통과하지 못합니다. 모든 인용이 유효한지 검사한 뒤에야 표시할 출처를 고릅니다.</p>
</details>


## 돌아보기

동료에게 재현 질문, 처음 어긋난 값, 같은 입력으로 확인한 수정 결과를 전달하세요. 이 기록이 있으면 프로젝트 팀은 검색과 화면 중 어디를 고칠지 합의할 수 있습니다.


## 고정한 질문으로 팀원의 첫 RAG를 인수하기

팀원이 보낸 서비스가 답변에 근거를 붙였다고 합니다. 앞의 인용 오류를 고친 뒤, 검색·답변·화면 출처가 같은 문서를 가리키는지 작은 인수 시험으로 확인합니다.

### 셀 06 · 출처 레코드만 바뀐 오류를 검출하기

동료가 보낸 오류 화면을 그대로 재현하려고 아래에 답변과 출처를 따로 적어 두었습니다. 앞 과제를 고친 뒤에도 같은 사례로 시작할 수 있게 이 기록과 새 검색 결과를 비교합니다.

In [4]:
reported_view = {"answer": "도서 대출 기간은 14일입니다. [library-1]",
                 "source": next(row for row in records if row["id"] == "library-2")}
actual_hits = retrieve("대출 기간은?", 1)
expected_id = actual_hits[0]["id"]
print(expected_id, reported_view["source"]["id"])
print(f"[{expected_id}]" in reported_view["answer"])

library-1 library-2
True


검색 결과와 답변에는 library-1이 들어 있지만 화면의 source는 library-2입니다. 팀원에게 “RAG가 틀렸다”고만 전하면 검색기를 다시 고칠 수 있습니다. 화면 조립 단계의 ID를 함께 보여 주세요.

### 과제 07 · 답변 가능·불가 질문을 같은 인터페이스로 인수 시험하기

질문을 받으면 retrieve를 한 번 호출해 answer와 sources를 반환하는 함수를 작성하세요. 대출 질문은 답 안의 인용과 첫 출처 ID가 일치해야 하고, 주차 질문은 빈 목록과 안내 문장을 반환해야 합니다.

In [5]:
# build_view(question)를 작성한 뒤 대출과 주차 두 입력으로 검산하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def build_view(question):
    docs = retrieve(question, 1)
    if not docs:
        return {&quot;answer&quot;: &quot;자료를 확인할 수 없습니다.&quot;, &quot;sources&quot;: []}
    first = docs[0]
    return {&quot;answer&quot;: f&quot;{first[&#x27;text&#x27;]} [{first[&#x27;id&#x27;]}]&quot;, &quot;sources&quot;: [first]}

loan_view = build_view(&quot;대출 기간은?&quot;)
citation = loan_view[&quot;sources&quot;][0][&quot;id&quot;]
assert citation == &quot;library-1&quot; and f&quot;[{citation}]&quot; in loan_view[&quot;answer&quot;]
empty_view = build_view(&quot;주차 요금은?&quot;)
assert empty_view[&quot;sources&quot;] == []
print(loan_view[&quot;answer&quot;])
print(empty_view[&quot;answer&quot;], empty_view[&quot;sources&quot;])</code></pre>
<p>실행 결과</p><pre><code>도서 대출 기간은 14일입니다. [library-1]
자료를 확인할 수 없습니다. []</code></pre>
<p>첫 출력은 대출 근거의 본문과 실제 문서 ID를, 두 번째는 안내와 빈 출처를 보여 줍니다. 검색 함수나 화면을 교체한 뒤에도 이 두 입력을 다시 실행하면 어느 약속을 깨뜨렸는지 빠르게 좁힐 수 있습니다.</p>
</details>

<mark>답변, 검색 기록, 표시한 출처가 모두 같은 요청을 설명하는지 확인하세요.</mark> 화면에 그럴듯한 자료명이 보이는 것만으로는 인수 시험을 마칠 수 없습니다. 대출 답변 옆에 휴관 안내가 붙은 앞 예제를 떠올리세요.

43장에서 팀이 고른 서비스에도 답변 가능한 질문 한 건과 보류할 질문 한 건을 남기세요.